# 4.0.2-Normalization and Dimentionality reduction

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
suppressMessages(suppressWarnings({
library(Seurat)
library(Signac)
library(tidyverse)
library(here)
library(glue)
}))

## Parameters

In [ ]:
#here::dr_here(show_reason = TRUE)
source(here("utils/bin.R"))
source(here("utils/paths.R"))
source(here("AVGUSTINOVA_02_to_30/utils/style.R"))
source(here("AVGUSTINOVA_02_to_30/utils/variables.R"))
set.seed(0)

In [ ]:
args <- commandArgs(TRUE)
op <- args[2]

## Load Data

In [ ]:
combined_obj <- readRDS(here(glue("{int}/{robj_dir}/4.0.0-combined_object_unify.rds")))

In [ ]:
hvgs_options <- readRDS(here(glue("{int}/{robj_dir}/4.0.1-hvf_list.rds")))
length(hvgs_options[[op]])

## ATAC unified

In [ ]:
DefaultAssay(combined_obj) <- "ATAC_unified"
combined_obj <- RunTFIDF(combined_obj)
combined_obj <- RunSVD(combined_obj,features=hvgs_options[["ATAC"]], reduction.name = "lsi.common")

In [ ]:
dim(Embeddings(combined_obj, reduction = "lsi.common"))

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)
DepthCor(combined_obj, reduction = "lsi.common", n=40)

In [ ]:
ElbowPlot(combined_obj,reduction="lsi.common", ndims=50)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 8)
DimPlot(combined_obj, reduction = "lsi.common", group.by = "patient", 
        dims=c(2,3), cols=colors_patients,raster=FALSE)

## RNA

In [ ]:
DefaultAssay(combined_obj) <- "RNA"

VariableFeatures(combined_obj) <- hvgs_options[[op]]
 
combined_obj <- combined_obj %>% 
  NormalizeData(normalization.method = "LogNormalize",
                scale.factor = 1e4) %>% 
  ScaleData(features= hvgs_options[[op]]) %>% 
  RunPCA(features= hvgs_options[[op]]) 

In [ ]:
ElbowPlot(combined_obj, ndims=50)

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 6)
FeaturePlot(combined_obj, reduction = "pca", features = c("nCount_RNA", "nFeature_RNA"),raster=FALSE)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
VizDimLoadings(combined_obj, dims = 1:5, reduction = "pca", ncol=5, nfeatures=50, balanced=TRUE)

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 5)
FeaturePlot(combined_obj, reduction = "pca", 
            features = c("MEOX2", "ERBB4", "PTPRC"), 
            order=T, ncol=3,raster=FALSE)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 6)
DimPlot(combined_obj, reduction = "pca", group.by = "patient", 
        cols= colors_patients, raster=FALSE)

## PCA regression

In [ ]:
df_info <- read_csv(here("utils/info_samples.csv"),show_col_types = FALSE)
df_clinical <- df_info[df_info$modality == "multiome",]
df_clinical <- df_clinical[,c("patient", "subproject", "quality", "location", "age_simple", 
                              "methylome", "alteration_status", "treatment_status")]

In [ ]:
df_tmp <- combined_obj@meta.data %>%
    rownames_to_column("barcodes") %>%
    left_join(df_clinical, by = "patient") %>%
    column_to_rownames("barcodes")

In [ ]:
#integrated_obj@meta.data <- df_tmp
copy_obj <- combined_obj
copy_obj@meta.data <- df_tmp

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6)

pcr_plot <- .plotPcr(
   obj = copy_obj, 
   var_list = covariates, 
   color_palette = cov_colors, 
   reduction = "pca",
   comp = 1:20
 )
pcr_plot

In [ ]:
remove(copy_obj)
remove(df_clinical)

## Save

In [ ]:
saveRDS(combined_obj, here(glue("{int}/{robj_dir}/4.0.2-combined_object_{op}.rds")))

In [ ]:
sessionInfo()